# EcoData | Organização e Qualidade de Dados Ambientais

**Python, Automação e Pandas.**

## 1. Bibliotecas utilizadas

Esta etapa importa as ferramentas necessárias ao processamento:

- **pathlib:** manipulação de caminhos, arquivos e diretórios.
- **pandas:** leitura, conversão e análise de dados em CSV e Excel.
- **hashlib:** cálculo de assinaturas SHA-256 para identificar arquivos idênticos.
- **shutil:** movimentação de arquivos para a pasta de arquivamento.

**Nenhum arquivo é modificado nesta etapa.**

In [19]:
from pathlib import Path
import pandas as pd
import hashlib
import shutil

## 2. Configuração dos diretórios

Esta etapa identifica a pasta principal do projeto e define os caminhos utilizados nas próximas operações.

**Diretórios principais:**

- `dados_recebidos`: armazena os arquivos originais, que devem permanecer intactos.
- `EcoData_Organizado`: recebe os arquivos classificados e padronizados em CSV.
- `EcoData_Duplicados_Arquivados`: guarda cópias redundantes durante o tratamento de duplicatas.

O programa utiliza caminhos relativos ao projeto para evitar dependência de um diretório absoluto.

**Resultado esperado:** apresentar os caminhos encontrados e confirmar a localização da pasta de origem.

A configuração dos caminhos não modifica os arquivos.

In [20]:
pasta_execucao = Path.cwd().resolve()

if (pasta_execucao / "dados_recebidos").is_dir():
    pasta_projeto = pasta_execucao
else:
    projetos_candidatos = [
        item for item in pasta_execucao.iterdir()
        if item.is_dir() and (item / "dados_recebidos").is_dir()
    ]

    if len(projetos_candidatos) != 1:
        raise FileNotFoundError(
            "Não foi possível identificar uma única pasta do EcoData a partir de "
            f"{pasta_execucao}. Execute o notebook na pasta do projeto ou em sua pasta pai."
        )

    pasta_projeto = projetos_candidatos[0].resolve()

pasta_origem = pasta_projeto / "dados_recebidos"
pasta_destino = pasta_projeto / "EcoData_Organizado"

if not pasta_origem.is_dir():
    raise FileNotFoundError(f"Pasta de origem não encontrada: {pasta_origem}")

print(f"Pasta do projeto: {pasta_projeto}")
print(f"Origem: {pasta_origem}")
print(f"Destino: {pasta_destino}")


Pasta do projeto: C:\Users\giova\Documents\Python\Projetos - Pandas, ML e Programas\6 - Automação e Organização de Dados Ambientais
Origem: C:\Users\giova\Documents\Python\Projetos - Pandas, ML e Programas\6 - Automação e Organização de Dados Ambientais\dados_recebidos
Destino: C:\Users\giova\Documents\Python\Projetos - Pandas, ML e Programas\6 - Automação e Organização de Dados Ambientais\EcoData_Organizado


## 3. Leitura e classificação dos arquivos

Nesta etapa, o programa define duas funções auxiliares:

**`listar_arquivos()`**

Percorre a pasta de origem e identifica arquivos nos formatos CSV e XLSX.

**`identificar_categoria()`**

Analisa as palavras presentes nos nomes dos arquivos e determina sua categoria:

| Categoria | Identificadores |
|---|---|
| Chuvas | chuva, precipitacao, pluviometria |
| Temperatura | temperatura, temp, termometria |
| Qualidade do Ar | ar, poluicao, pm25; qualidade_ar contém o token ar |
| Pendências | Arquivos não reconhecidos ou com pistas conflitantes |

A classificação utiliza tokens completos do nome, normalizando maiúsculas/minúsculas, hífens e underscores. A palavra `qualidade` isolada não identifica qualidade do ar. Pistas de várias categorias, ou de ar junto de água/solo, seguem para `Pendencias` com aviso. O conteúdo interno não é usado para escolher a categoria.

**Importante:** essas funções apenas identificam informações. Não movem nem alteram os documentos.

In [21]:
def listar_arquivos(pasta_origem):
    pasta_origem = Path(pasta_origem)

    if not pasta_origem.is_dir():
        raise FileNotFoundError(
            f"Pasta de origem não encontrada: {pasta_origem}"
        )

    arquivos = []

    for arquivo in pasta_origem.iterdir():
        if arquivo.is_file() and arquivo.suffix.lower() in (".csv", ".xlsx"):
            arquivos.append(arquivo)

    return sorted(arquivos)

def identificar_categoria(arquivo):
    nome = arquivo.stem.lower().replace("-", "_")
    palavras = set(nome.split("_"))

    indicios = {
        "Chuvas": {"chuva", "precipitacao", "pluviometria"},
        "Temperatura": {"temperatura", "temp", "termometria"},
        "Qualidade_do_Ar": {"ar", "poluicao", "pm25"}
    }
    categorias = [
        categoria for categoria, termos in indicios.items()
        if palavras.intersection(termos)
    ]

    # Água e solo não possuem categoria no contrato atual. A palavra genérica
    # "qualidade" nunca basta; pistas de ar junto desses temas exigem revisão.
    outro_tema = (
        "Qualidade_do_Ar" in categorias
        and bool(palavras.intersection({"agua", "solo"}))
    )
    if len(categorias) > 1 or outro_tema:
        motivo = (
            "pistas de mais de uma categoria: " + ", ".join(categorias)
            if len(categorias) > 1
            else "pistas de qualidade do ar junto de água ou solo"
        )
        print(
            f"[CLASSIFICAÇÃO AMBÍGUA] {arquivo.name}: {motivo}. "
            "Encaminhado para Pendencias; confira o nome original."
        )
        return "Pendencias"

    return categorias[0] if categorias else "Pendencias"


### 4. Cálculo de hash, conversão e organização dos arquivos

Nesta etapa, o programa realiza o cálculo de hash dos arquivos, converte os documentos recebidos para o formato CSV e organiza os resultados nas pastas correspondentes às categorias ambientais.

A função `calcular_hash()` utiliza o algoritmo SHA-256 para gerar uma assinatura digital baseada no conteúdo de cada arquivo. Essa assinatura permite comparar documentos e identificar arquivos com conteúdo binário idêntico.

A função `converter_para_csv()` utiliza Pandas para ler CSV e a primeira aba de XLSX, preservando `id_registro` e `estacao_id` como texto. As outras colunas mantêm a inferência atual. A saída usa UTF-8-SIG, CRLF explícito e não inclui o índice do DataFrame. IDs Excel numéricos com formatação especial, datas ou fórmulas exigem conferência: a conversão é interrompida, sem inventar zeros.

A função `organizar_arquivos()` identifica a categoria de cada documento e direciona os arquivos para as pastas `Chuvas`, `Temperatura`, `Qualidade_do_Ar` e `Pendencias`.

Durante a organização, o programa usa SHA-256 de uma representação dos campos CSV que normaliza LF/CRLF. Essa assinatura difere do hash binário integral usado na rotina de duplicatas. O programa consulta as assinaturas dos arquivos ativos e anteriormente arquivados. Quando identifica que um conteúdo arquivado já possui uma cópia ativa, evita sua reimportação.

A função compara o CSV convertido com os destinos existentes. Conteúdos iguais são ignorados; conteúdos divergentes são preservados com nomes alternativos, sem sobrescrita. A publicação usa um arquivo temporário, e o resumo informa também os conflitos.

Quando o conteúdo difere apenas em IDs textuais numericamente equivalentes, a organização sinaliza **migração R5 pendente**. Essa semelhança não prova identidade: exige conferência humana, sem substituir o histórico nem publicar automaticamente uma nova cópia.

**Objetivo:** padronizar e organizar os arquivos ambientais, reduzir reprocessamentos desnecessários e evitar a recriação de duplicatas anteriormente arquivadas que ainda possuem uma cópia ativa.


In [22]:
# ==================================================
# CÁLCULO DE HASH SHA-256
# ==================================================

def calcular_hash(arquivo):
    hash_arquivo = hashlib.sha256()

    with open(arquivo, "rb") as arquivo_aberto:
        for bloco in iter(
            lambda: arquivo_aberto.read(8192),
            b""
        ):
            hash_arquivo.update(bloco)

    return hash_arquivo.hexdigest()


# ==================================================
# CONVERSÃO DE ARQUIVOS PARA CSV
# ==================================================

def converter_para_csv(arquivo, destino):
    arquivo = Path(arquivo)
    # Esquema observado: somente os identificadores são textuais.
    tipos_identificadores = {"id_registro": "string", "estacao_id": "string"}

    if arquivo.suffix.lower() == ".csv":
        dados = pd.read_csv(
            arquivo,
            encoding="utf-8-sig",
            dtype=tipos_identificadores
        )

    else:
        # Pandas usa openpyxl para XLSX. Conferimos a representação armazenada
        # antes de converter: um número formatado como 000 não é texto "001".
        from openpyxl import load_workbook

        planilha = load_workbook(arquivo, read_only=True, data_only=False)
        try:
            aba = planilha.worksheets[0]
            cabecalho = next(aba.iter_rows(min_row=1, max_row=1), ())
            identificadores = {
                indice: celula.value
                for indice, celula in enumerate(cabecalho)
                if celula.value in tipos_identificadores
            }
            for linha in aba.iter_rows(min_row=2):
                for indice, nome in identificadores.items():
                    celula = linha[indice]
                    if celula.value is None:
                        continue
                    formato_ambiguo = (
                        celula.data_type == "n"
                        and celula.number_format not in ("General", "0")
                    )
                    if formato_ambiguo or celula.data_type in ("d", "f"):
                        raise ValueError(
                            f"IDENTIFICADOR AMBÍGUO: {arquivo.name}, "
                            f"{aba.title}!{celula.coordinate}, coluna {nome}; "
                            f"tipo {celula.data_type}, formato {celula.number_format!r}. "
                            "Confirme o esquema e prepare uma cópia com o ID textual; "
                            "nenhum zero será inventado."
                        )
        finally:
            planilha.close()

        dados = pd.read_excel(arquivo, dtype=tipos_identificadores)

    dados.to_csv(
        destino,
        index=False,
        encoding="utf-8-sig",
        lineterminator="\r\n"
    )


# ==================================================
# ORGANIZAÇÃO DOS ARQUIVOS
# ==================================================

def organizar_arquivos(pasta_origem, pasta_destino):
    import csv
    import io
    import os
    import tempfile

    pasta_destino = Path(pasta_destino)
    pasta_backup = pasta_destino.parent / "EcoData_Duplicados_Arquivados"
    categorias = ["Chuvas", "Temperatura", "Qualidade_do_Ar", "Pendencias"]
    contagem = {categoria: 0 for categoria in categorias}
    ignorados = 0
    conflitos = 0
    migracoes = 0
    erros = 0

    for categoria in categorias:
        (pasta_destino / categoria).mkdir(parents=True, exist_ok=True)

    # Representa os campos CSV de forma estável entre LF e CRLF.
    # Mantém diferenças nos textos, na ordem das linhas e nos identificadores.
    def assinatura_csv(arquivo):
        buffer = io.StringIO(newline="")
        with Path(arquivo).open("r", encoding="utf-8-sig", newline="") as entrada:
            leitor = csv.reader(entrada, strict=True)
            cabecalho = next(leitor, None)
            if not cabecalho:
                raise ValueError("CSV vazio ou sem cabeçalho")
            escritor = csv.writer(buffer, lineterminator="\n")
            escritor.writerow(cabecalho)
            for linha in leitor:
                escritor.writerow(linha)
        return hashlib.sha256(buffer.getvalue().encode("utf-8")).hexdigest()

    def possivel_migracao_r5(esperado, existente):
        # Apenas sinaliza equivalência numérica de IDs textuais divergentes.
        # Não converte arquivos históricos nem os declara equivalentes.
        from decimal import Decimal, InvalidOperation

        tabelas = []
        for caminho in (esperado, existente):
            with Path(caminho).open("r", encoding="utf-8-sig", newline="") as entrada:
                tabelas.append(list(csv.reader(entrada, strict=True)))
        novo, antigo = tabelas
        if not novo or not antigo or novo[0] != antigo[0] or len(novo) != len(antigo):
            return False
        ids = {
            indice for indice, nome in enumerate(novo[0])
            if nome in ("id_registro", "estacao_id")
        }
        mudou = False
        for linha_nova, linha_antiga in zip(novo[1:], antigo[1:]):
            if len(linha_nova) != len(novo[0]) or len(linha_antiga) != len(novo[0]):
                return False
            for indice, (atual, anterior) in enumerate(zip(linha_nova, linha_antiga)):
                if atual == anterior:
                    continue
                if indice not in ids:
                    return False
                try:
                    valor_atual, valor_anterior = Decimal(atual), Decimal(anterior)
                except InvalidOperation:
                    return False
                if not (valor_atual.is_finite() and valor_anterior.is_finite()
                        and valor_atual == valor_anterior):
                    return False
                mudou = True
        return mudou

    arquivos_invalidos = []

    def hashes_csv_validos(pasta):
        hashes = set()
        if pasta.is_dir():
            for arquivo_csv in pasta.rglob("*.csv"):
                try:
                    hashes.add(assinatura_csv(arquivo_csv))
                except Exception as erro:
                    arquivos_invalidos.append((arquivo_csv, erro))
        return hashes

    hashes_arquivados = hashes_csv_validos(pasta_backup)
    hashes_ativos = hashes_csv_validos(pasta_destino)
    erros += len(arquivos_invalidos)
    for arquivo_invalido, erro in arquivos_invalidos:
        print(f"ERRO ao comparar {arquivo_invalido}: {erro}")

    for arquivo in listar_arquivos(pasta_origem):
        categoria = identificar_categoria(arquivo)
        destino = pasta_destino / categoria / f"{arquivo.stem}.csv"
        temporario = None

        try:
            # O arquivo temporário fica no mesmo diretório do destino.
            # No Windows, ele precisa estar fechado antes da conversão.
            with tempfile.NamedTemporaryFile(
                prefix=".ecodata_", suffix=".tmp",
                dir=destino.parent, delete=False
            ) as arquivo_temporario:
                temporario = Path(arquivo_temporario.name)

            converter_para_csv(arquivo, temporario)
            assinatura = assinatura_csv(temporario)

            conflito = False
            motivo = "conteúdo divergente ou incompleto"
            if os.path.lexists(destino):
                try:
                    assinatura_destino = assinatura_csv(destino)
                except Exception as erro_destino:
                    conflito = True
                    motivo = f"destino inválido ou ilegível ({erro_destino})"
                else:
                    if assinatura_destino == assinatura:
                        hashes_ativos.add(assinatura)
                        ignorados += 1
                        print(f"IGNORADO (já existe e foi validado): {destino.name}")
                        continue
                    conflito = True

            # Reconhece uma saída alternativa criada em execução anterior.
            alternativos = sorted(set(
                list(destino.parent.glob(f"{destino.stem}_origem_*.csv")) +
                list(destino.parent.glob(f"{destino.stem}_recuperado*.csv"))
            ))
            ja_preservado = None
            for alternativo in alternativos:
                try:
                    if assinatura_csv(alternativo) == assinatura:
                        ja_preservado = alternativo
                        break
                except Exception:
                    continue

            if ja_preservado is not None:
                if conflito:
                    conflitos += 1
                    print(
                        f"CONFLITO: origem {arquivo.name}; destino {destino.name} "
                        f"({motivo}); conteúdo já preservado em "
                        f"{ja_preservado.name}. Verifique o destino conflitante."
                    )
                else:
                    print(
                        f"IGNORADO (já presente em nome alternativo): "
                        f"{arquivo.name} -> {ja_preservado.name}"
                    )
                hashes_ativos.add(assinatura)
                ignorados += 1
                continue

            # Preserva EC-02: um conteúdo arquivado só é ignorado
            # quando outra cópia válida já está ativa.
            if assinatura in hashes_arquivados and assinatura in hashes_ativos:
                if conflito:
                    conflitos += 1
                    print(
                        f"CONFLITO: origem {arquivo.name}; destino {destino.name} "
                        f"({motivo}). Verifique o destino conflitante."
                    )
                ignorados += 1
                print(f"IGNORADO (conteúdo já presente na base ativa): {arquivo.name}")
                continue

            # Se a classificação mudou, preserve a cópia ativa anterior
            # e sinalize a migração antes de criar outra cópia.
            copia_outra_categoria = None
            for categoria_anterior in categorias:
                if categoria_anterior == categoria:
                    continue
                anterior = pasta_destino / categoria_anterior / destino.name
                if not anterior.is_file():
                    continue
                try:
                    if assinatura_csv(anterior) == assinatura:
                        copia_outra_categoria = anterior
                        break
                except (OSError, UnicodeError, csv.Error, ValueError):
                    continue
            if copia_outra_categoria is not None:
                conflitos += 1
                print(
                    f"[MIGRAÇÃO DE CATEGORIA PENDENTE] Origem: {arquivo.name}; "
                    f"cópia ativa: {copia_outra_categoria}; "
                    f"categoria atual: {categoria}. "
                    "Nenhum arquivo foi publicado; revise a classificação "
                    "antes de migrar a cópia ativa."
                )
                continue

            # IDs preservados podem divergir de conversões históricas que
            # perderam zeros. Esses casos exigem migração autorizada, sem
            # publicar automaticamente uma segunda versão do mesmo conjunto.
            historicos = sorted(pasta_destino.rglob("*.csv"))
            if assinatura in hashes_ativos and pasta_backup.is_dir():
                historicos += sorted(pasta_backup.rglob("*.csv"))
            migracao_pendente = None
            for historico in historicos:
                try:
                    if possivel_migracao_r5(temporario, historico):
                        migracao_pendente = historico
                        break
                except (OSError, UnicodeError, csv.Error, ValueError):
                    continue
            if migracao_pendente is not None:
                migracoes += 1
                conflitos += 1
                print(
                    f"[MIGRAÇÃO R5 NECESSÁRIA] Origem: {arquivo.name}; "
                    f"cópia histórica: {migracao_pendente}. "
                    "Diferença restrita a identificadores numericamente equivalentes. "
                    "Confirme os originais antes de autorizar a migração; "
                    "nenhum arquivo foi publicado ou substituído."
                )
                continue

            if conflito:
                if assinatura in hashes_arquivados:
                    nome_alternativo = f"{destino.stem}_recuperado"
                else:
                    nome_alternativo = (
                        f"{destino.stem}_origem_{arquivo.suffix.lower().lstrip('.')}"
                    )
                candidato = destino.with_name(f"{nome_alternativo}.csv")
            else:
                nome_alternativo = None
                candidato = destino
            numero = 2

            while True:
                if os.path.lexists(candidato):
                    try:
                        assinatura_candidato = assinatura_csv(candidato)
                    except Exception:
                        assinatura_candidato = None
                    if assinatura_candidato == assinatura:
                        hashes_ativos.add(assinatura)
                        ignorados += 1
                        if candidato != destino:
                            conflitos += 1
                            print(
                                f"CONFLITO: origem {arquivo.name}; destino "
                                f"{destino.name} ({motivo}); conteúdo já preservado "
                                f"em {candidato.name}. Verifique o destino conflitante."
                            )
                        else:
                            print(f"IGNORADO (já existe e foi validado): {destino.name}")
                        break
                    if candidato == destino:
                        conflito = True
                        motivo = "conteúdo divergente ou incompleto"
                        nome_alternativo = (
                            f"{destino.stem}_origem_{arquivo.suffix.lower().lstrip('.')}"
                        )
                        candidato = destino.with_name(f"{nome_alternativo}.csv")
                    else:
                        candidato = destino.with_name(
                            f"{nome_alternativo}_{numero}.csv"
                        )
                        numero += 1
                    continue

                try:
                    # Um hard link no mesmo volume publica sem substituir
                    # um destino criado por outra execução.
                    os.link(temporario, candidato)
                except FileExistsError:
                    continue

                hashes_ativos.add(assinatura)
                contagem[categoria] += 1
                if conflito:
                    conflitos += 1
                    print(
                        f"CONFLITO: origem {arquivo.name}; destino {destino.name} "
                        f"({motivo}); nova saída {categoria}/{candidato.name}. "
                        "Verifique o destino conflitante."
                    )
                else:
                    print(f"{arquivo.name}  ->  {categoria}/{candidato.name}")
                break

        except Exception as erro:
            erros += 1
            print(f"ERRO ao processar {arquivo.name}: {erro}")
        finally:
            if temporario is not None and temporario.exists():
                try:
                    temporario.unlink()
                except OSError as erro_limpeza:
                    erros += 1
                    print(f"ERRO ao remover temporário {temporario.name}: {erro_limpeza}")

    print("\nRESUMO DA ORGANIZAÇÃO")
    for categoria, total in contagem.items():
        print(f"{categoria}: {total}")
    print(f"Ignorados (existentes ou redundantes): {ignorados}")
    print(f"Conflitos: {conflitos}")
    print(f"Migrações R5 pendentes: {migracoes}")
    print(f"Erros: {erros}")


# ==================================================
# EXECUÇÃO
# ==================================================

organizar_arquivos(
    pasta_origem,
    pasta_destino
)


IGNORADO (já existe e foi validado): ar_copacabana_2026-10-18.csv
IGNORADO (já existe e foi validado): arquivo_sem_identificacao.csv
IGNORADO (já existe e foi validado): chuva_bangu_2026-10-16.csv
IGNORADO (já existe e foi validado): chuva_meier_2026-10-07.csv
IGNORADO (já existe e foi validado): chuva_tijuca_2026-10-03.csv
IGNORADO (já existe e foi validado): coleta_turno_b.csv
IGNORADO (já existe e foi validado): dados_estacao_04.csv
IGNORADO (já existe e foi validado): pluviometria_copacabana_2026-10-14.csv
IGNORADO (já existe e foi validado): pm25_tijuca_2026-10-11.csv
IGNORADO (conteúdo já presente na base ativa): poluicao_centro_2026-10-05_repasse.csv
IGNORADO (já existe e foi validado): poluicao_madureira_2026-10-20.csv
IGNORADO (já existe e foi validado): precipitacao_centro_2026-10-10.csv
IGNORADO (já existe e foi validado): precipitacao_madureira_2026-10-19.csv
IGNORADO (conteúdo já presente na base ativa): precipitacao_tijuca_2026-10-03_repasse.csv
IGNORADO (já existe e foi 

### 5. Identificação de arquivos duplicados

Nesta etapa, o programa verifica a existência de arquivos duplicados dentro da pasta de dados ambientais organizados.

A função `verificar_arquivos_duplicados()` percorre os arquivos CSV das diferentes categorias e utiliza a função `calcular_hash()`, definida na etapa anterior, para comparar suas assinaturas SHA-256.

Quando dois arquivos apresentam o mesmo hash, o programa identifica o conteúdo duplicado e informa os nomes dos documentos envolvidos, mesmo que possuam nomes diferentes.

Ao final da verificação, é apresentado um relatório com a quantidade total de arquivos analisados, o número de arquivos duplicados e a quantidade de conteúdos únicos encontrados.

Esta etapa realiza somente a identificação das duplicatas, sem excluir ou movimentar arquivos. O tratamento e o arquivamento das cópias redundantes são realizados em uma etapa específica do programa.

**Objetivo:** identificar conteúdos repetidos na base organizada, evitando que informações redundantes sejam consideradas registros independentes em análises posteriores.


In [23]:
# ==================================================
# VERIFICAÇÃO DE ARQUIVOS DUPLICADOS
# ==================================================

def verificar_arquivos_duplicados(pasta_destino):

    print("\nVERIFICAÇÃO DE ARQUIVOS DUPLICADOS")
    print("=" * 50)

    pasta = Path(pasta_destino)

    arquivos = sorted(
        pasta.rglob("*.csv")
    )

    hashes = {}
    duplicados = []

    for arquivo in arquivos:

        assinatura = calcular_hash(arquivo)

        if assinatura in hashes:

            arquivo_original = hashes[assinatura]

            duplicados.append(
                (arquivo_original, arquivo)
            )

            print("\nDUPLICATA ENCONTRADA")

            print(
                f"Arquivo original: "
                f"{arquivo_original.name}"
            )

            print(
                f"Arquivo duplicado: "
                f"{arquivo.name}"
            )

        else:

            hashes[assinatura] = arquivo

    print("\nRESUMO DA VERIFICAÇÃO")
    print("-" * 50)

    print(
        f"Arquivos analisados: {len(arquivos)}"
    )

    print(
        f"Arquivos duplicados: {len(duplicados)}"
    )

    print(
        f"Arquivos únicos: {len(hashes)}"
    )

    if not duplicados:
        print(
            "\nNenhum arquivo duplicado encontrado."
        )

    return duplicados


# ==================================================
# EXECUÇÃO
# ==================================================

arquivos_duplicados = verificar_arquivos_duplicados(
    pasta_destino
)



VERIFICAÇÃO DE ARQUIVOS DUPLICADOS

RESUMO DA VERIFICAÇÃO
--------------------------------------------------
Arquivos analisados: 22
Arquivos duplicados: 0
Arquivos únicos: 22

Nenhum arquivo duplicado encontrado.


## 6. Tratamento de arquivos duplicados

Após identificar documentos com conteúdo idêntico, esta etapa permite retirar cópias redundantes da base de análise.

A função `remover_arquivos_duplicados()` possui dois modos:

**Simulação — `executar=False`**

Identifica as duplicatas e informa o tratamento previsto, sem modificar os arquivos.

**Execução — `executar=True`**

Retira as cópias redundantes de `EcoData_Organizado`.

Quando ainda não existe uma cópia idêntica no backup, o arquivo é movido para `EcoData_Duplicados_Arquivados`.

Quando já existe um backup com o mesmo SHA-256, a cópia redundante da pasta organizada é excluída após a confirmação do conteúdo arquivado, sem criar outro backup idêntico.

O arquivo original em `dados_recebidos` é preservado.

**Verificação de sucesso**

Após uma remoção efetiva, o programa pode executar novamente a verificação SHA-256.

Na situação esperada, a pasta organizada permanece com 22 arquivos e nenhuma duplicata exata.

**Atenção:** mantenha `executar=False` como configuração padrão para evitar remoções involuntárias ao utilizar a opção "Executar Todas as Células".


In [24]:
def remover_arquivos_duplicados(pasta_destino, executar=False):

    pasta_destino = Path(pasta_destino)

    if not pasta_destino.is_dir():
        raise FileNotFoundError(
            f"Pasta não encontrada: {pasta_destino}"
        )

    pasta_backup = (
        pasta_destino.parent / "EcoData_Duplicados_Arquivados"
    )

    arquivos = sorted(pasta_destino.rglob("*.csv"))

    grupos = {}
    duplicados = []
    backups_por_hash = {}

    # Verifica os arquivos que já estão arquivados
    if pasta_backup.is_dir():

        for arquivo_backup in sorted(pasta_backup.rglob("*.csv")):

            assinatura = calcular_hash(arquivo_backup)

            backups_por_hash.setdefault(
                assinatura,
                arquivo_backup
            )

    # Agrupa os arquivos organizados pelo conteúdo
    for arquivo in arquivos:

        assinatura = calcular_hash(arquivo)

        grupos.setdefault(
            assinatura,
            []
        ).append(arquivo)

    print("\nREMOÇÃO DE ARQUIVOS DUPLICADOS")
    print("=" * 50)

    for assinatura, grupo in grupos.items():

        if len(grupo) < 2:
            continue

        # Prioriza manter arquivos sem "repasse"
        grupo.sort(
            key=lambda arquivo: (
                "repasse" in arquivo.stem.lower(),
                str(arquivo).lower()
            )
        )

        arquivo_mantido = grupo[0]

        for arquivo_duplicado in grupo[1:]:

            duplicados.append(arquivo_duplicado)

            print(f"\nArquivo mantido: {arquivo_mantido.name}")
            print(f"Duplicata: {arquivo_duplicado.name}")

            # Verifica se o conteúdo já existe no backup
            if assinatura in backups_por_hash:

                arquivo_no_backup = backups_por_hash[assinatura]

                if executar:

                    # Confirma a integridade do backup
                    if (
                        not arquivo_no_backup.is_file()
                        or calcular_hash(arquivo_no_backup) != assinatura
                    ):
                        raise RuntimeError(
                            f"Backup não pôde ser confirmado: {arquivo_no_backup}"
                        )

                    # Remove somente a cópia redundante
                    # da pasta EcoData_Organizado
                    arquivo_duplicado.unlink()

                    print(
                        "Status: backup idêntico já existe; "
                        "cópia redundante removida da pasta organizada"
                    )

                else:

                    print(
                        "Status: backup idêntico já existe; "
                        "nenhuma alteração (simulação)"
                    )

                continue

            # Simulação sem modificar arquivos
            if not executar:

                print(
                    "Status: identificado, sem alterações"
                )

                continue

            # Arquiva apenas quando ainda não existe
            # uma cópia idêntica no backup
            destino_backup = (
                pasta_backup /
                arquivo_duplicado.relative_to(pasta_destino)
            )

            destino_backup.parent.mkdir(
                parents=True,
                exist_ok=True
            )

            destino_final = destino_backup
            contador = 2

            # Evita sobrescrever nomes já existentes
            while destino_final.exists():

                destino_final = destino_backup.with_name(
                    f"{destino_backup.stem}_{contador}"
                    f"{destino_backup.suffix}"
                )

                contador += 1

            shutil.move(
                str(arquivo_duplicado),
                str(destino_final)
            )

            # Atualiza o controle dos backups
            backups_por_hash[assinatura] = destino_final

            print("Status: movido para backup")

    print("\nRESUMO")
    print("-" * 50)

    print(f"Arquivos analisados: {len(arquivos)}")
    print(f"Duplicatas encontradas: {len(duplicados)}")

    if executar:

        restantes = len(
            list(pasta_destino.rglob("*.csv"))
        )

        print(f"Arquivos restantes: {restantes}")

    else:

        print(
            "Modo de simulação: "
            "nenhum arquivo foi movido ou removido"
        )

    return duplicados


# EXECUÇÃO DA FUNÇÃO

duplicados = remover_arquivos_duplicados(
    pasta_destino,
    executar=False # Selecionar True para remover arquivos duplicados ou False para simular a remoção 
)

duplicados_restantes = verificar_arquivos_duplicados(
    pasta_destino
)

# Apresenta a mensagem conforme o resultado
if len(duplicados_restantes) == 0:

    print("\nSUCESSO!")
    print("A base EcoData está livre de arquivos duplicados exatos.")

else:

    print("\nATENÇÃO!")
    print(f"Foram encontrados {len(duplicados_restantes)} arquivos duplicados.")


REMOÇÃO DE ARQUIVOS DUPLICADOS

RESUMO
--------------------------------------------------
Arquivos analisados: 22
Duplicatas encontradas: 0
Modo de simulação: nenhum arquivo foi movido ou removido

VERIFICAÇÃO DE ARQUIVOS DUPLICADOS

RESUMO DA VERIFICAÇÃO
--------------------------------------------------
Arquivos analisados: 22
Arquivos duplicados: 0
Arquivos únicos: 22

Nenhum arquivo duplicado encontrado.

SUCESSO!
A base EcoData está livre de arquivos duplicados exatos.


## 7. Inspeção inicial da qualidade dos dados

Esta etapa utiliza Pandas para examinar a estrutura de cada arquivo CSV organizado.

A função `verificar_qualidade()` apresenta:

- Nome do arquivo e categoria.
- Quantidade de registros.
- Nomes das colunas.
- Quantidade de valores ausentes.
- Quantidade de linhas duplicadas internamente.

**Limitação importante**

O método `dados.duplicated()` identifica registros repetidos dentro de um mesmo DataFrame.

Ele não compara diretamente os registros de diferentes arquivos.

A verificação não altera nenhuma informação e serve como diagnóstico inicial da qualidade das bases.


In [25]:
def verificar_qualidade(pasta_destino):
    pasta_destino = Path(pasta_destino)

    print("\nRELATÓRIO INICIAL DE QUALIDADE")
    print("=" * 50)

    categorias = [
        "Chuvas",
        "Temperatura",
        "Qualidade_do_Ar",
        "Pendencias"
    ]

    for categoria in categorias:
        pasta = pasta_destino / categoria

        for arquivo in sorted(pasta.glob("*.csv")):
            try:
                dados = pd.read_csv(
                    arquivo, encoding="utf-8-sig",
                    dtype={"id_registro": "string", "estacao_id": "string"}
                )

                print(f"\nArquivo: {arquivo.name}")
                print(f"Categoria: {categoria}")
                print(f"Linhas: {len(dados)}")
                print(f"Colunas: {list(dados.columns)}")

                ausentes = dados.isna().sum().sum()
                duplicados = dados.duplicated().sum()

                print(f"Valores ausentes: {ausentes}")
                print(f"Linhas duplicadas: {duplicados}")

            except Exception as erro:
                print(f"Erro ao ler {arquivo.name}: {erro}")

verificar_qualidade(pasta_destino)



RELATÓRIO INICIAL DE QUALIDADE

Arquivo: chuva_bangu_2026-10-16.csv
Categoria: Chuvas
Linhas: 40
Colunas: ['id_registro', 'data_hora', 'estacao_id', 'bairro', 'precipitacao_mm', 'periodo_medicao']
Valores ausentes: 0
Linhas duplicadas: 0

Arquivo: chuva_meier_2026-10-07.csv
Categoria: Chuvas
Linhas: 38
Colunas: ['id_registro', 'data_hora', 'estacao_id', 'bairro', 'precipitacao_mm', 'periodo_medicao']
Valores ausentes: 2
Linhas duplicadas: 0

Arquivo: chuva_tijuca_2026-10-03.csv
Categoria: Chuvas
Linhas: 32
Colunas: ['id_registro', 'data_hora', 'estacao_id', 'bairro', 'precipitacao_mm', 'periodo_medicao']
Valores ausentes: 0
Linhas duplicadas: 0

Arquivo: pluviometria_copacabana_2026-10-14.csv
Categoria: Chuvas
Linhas: 30
Colunas: ['id_registro', 'data_hora', 'estacao_id', 'bairro', 'precipitacao_mm', 'periodo_medicao']
Valores ausentes: 0
Linhas duplicadas: 0

Arquivo: precipitacao_centro_2026-10-10.csv
Categoria: Chuvas
Linhas: 28
Colunas: ['id_registro', 'data_hora', 'estacao_id', '


Arquivo: pm25_tijuca_2026-10-11.csv
Categoria: Qualidade_do_Ar
Linhas: 28
Colunas: ['id_registro', 'data_hora', 'estacao_id', 'bairro', 'pm25_ug_m3', 'pm10_ug_m3']
Valores ausentes: 1
Linhas duplicadas: 0

Arquivo: poluicao_madureira_2026-10-20.csv
Categoria: Qualidade_do_Ar
Linhas: 26
Colunas: ['id_registro', 'data_hora', 'estacao_id', 'bairro', 'pm25_ug_m3', 'pm10_ug_m3']
Valores ausentes: 0
Linhas duplicadas: 0

Arquivo: qualidade_ar_centro_2026-10-05.csv
Categoria: Qualidade_do_Ar
Linhas: 32
Colunas: ['id_registro', 'data_hora', 'estacao_id', 'bairro', 'pm25_ug_m3', 'pm10_ug_m3']
Valores ausentes: 0
Linhas duplicadas: 0

Arquivo: qualidade_ar_meier_2026-10-15.csv
Categoria: Qualidade_do_Ar
Linhas: 36
Colunas: ['id_registro', 'data_hora', 'estacao_id', 'bairro', 'pm25_ug_m3', 'pm10_ug_m3']
Valores ausentes: 0
Linhas duplicadas: 0

Arquivo: arquivo_sem_identificacao.csv
Categoria: Pendencias
Linhas: 24
Colunas: ['id_registro', 'data_hora', 'estacao_id', 'bairro', 'temperatura_c', 'u

## 8. Investigação dos valores ausentes

Esta etapa identifica quais colunas e registros possuem informações faltantes, representadas pelo Pandas como `NaN`.

A função `detalhar_ausentes()` mostra o nome do arquivo, as colunas afetadas e as linhas que precisam ser investigadas.

**Resultados identificados na análise inicial:**

| Arquivo | Coluna afetada | Ausentes |
|---|---|---:|
| chuva_meier_2026-10-07.csv | precipitacao_mm | 2 |
| temp_bangu_2026-10-20.csv | umidade_relativa_pct | 1 |
| pm25_tijuca_2026-10-11.csv | pm25_ug_m3 | 1 |

**Total histórico: 4 valores ausentes.**

Um valor ausente não deve ser interpretado automaticamente como zero.

O tratamento depende do significado da variável e do objetivo da análise.

Esta etapa apenas identifica os problemas, preservando os dados originais.


In [26]:
def detalhar_ausentes(pasta_destino):
    pasta_destino = Path(pasta_destino)

    categorias = [
        "Chuvas",
        "Temperatura",
        "Qualidade_do_Ar",
        "Pendencias"
    ]

    print("\nINVESTIGAÇÃO DE VALORES AUSENTES")
    print("=" * 50)
    lidos = 0
    falhas = []

    for categoria in categorias:
        pasta = pasta_destino / categoria

        for arquivo in sorted(pasta.glob("*.csv")):
            try:
                dados = pd.read_csv(
                    arquivo, encoding="utf-8-sig",
                    dtype={"id_registro": "string", "estacao_id": "string"}
                )
            except UnicodeDecodeError as erro:
                falhas.append((categoria, arquivo.resolve()))
                print("\n[ERRO DE DECODIFICAÇÃO]")
                print(f"Arquivo: {arquivo.name}")
                print(f"Categoria: {categoria}")
                print(f"Caminho: {arquivo.resolve()}")
                print("Status: LEITURA NÃO CONCLUÍDA")
                print(f"Detalhes: {erro}")
                print("ORIENTAÇÃO: confira o formato do original e descubra sua codificação.")
                print("Crie uma cópia em UTF-8-SIG sem alterar o original; confira estrutura e registros.")
                print("Reprocesse a cópia pelo fluxo seguro do EcoData e confira a categoria.")
                print("Nenhuma alteração foi feita no arquivo. Continuando a análise...")
                continue
            except (pd.errors.ParserError, pd.errors.EmptyDataError, OSError) as erro:
                falhas.append((categoria, arquivo.resolve()))
                print("\n[ERRO DE LEITURA]")
                print(f"Arquivo: {arquivo.name}")
                print(f"Categoria: {categoria}")
                print(f"Caminho: {arquivo.resolve()}")
                print("Status: LEITURA NÃO CONCLUÍDA")
                print(f"Tipo: {type(erro).__name__}")
                print(f"Detalhes: {erro}")
                print("ORIENTAÇÃO: verifique a estrutura, o conteúdo e o acesso ao CSV.")
                print("Corrija uma cópia, valide sua leitura e reprocesse pelo fluxo seguro do EcoData.")
                print("Nenhuma alteração foi feita no arquivo. Continuando a análise...")
                continue

            lidos += 1

            ausentes = dados.isna().sum()

            if ausentes.sum() > 0:
                print(f"\nArquivo: {arquivo.name}")

                print("\nColunas com valores ausentes:")
                print(ausentes[ausentes > 0])

                print("\nRegistros afetados:")
                registros = dados[
                    dados.isna().any(axis=1)
                ]

                print(registros.to_string(index=False))

    print("\nRESUMO DA LEITURA")
    print(f"Arquivos lidos com sucesso: {lidos}")
    print(f"Arquivos com erro de leitura: {len(falhas)}")
    if falhas:
        print("Arquivos não analisados:")
        for categoria_falha, caminho_falha in falhas:
            print(f"- {categoria_falha}: {caminho_falha}")

detalhar_ausentes(pasta_destino)



INVESTIGAÇÃO DE VALORES AUSENTES

Arquivo: chuva_meier_2026-10-07.csv

Colunas com valores ausentes:
precipitacao_mm    2
dtype: int64

Registros afetados:
        id_registro        data_hora estacao_id bairro  precipitacao_mm periodo_medicao
PLU-ME-20261007-009 2026-10-07 04:00  PLU-ME-01  Méier              NaN          30 min
PLU-ME-20261007-022 2026-10-07 10:30  PLU-ME-01  Méier              NaN          30 min

Arquivo: temp_bangu_2026-10-20.csv

Colunas com valores ausentes:
umidade_relativa_pct    1
dtype: int64

Registros afetados:
        id_registro        data_hora estacao_id bairro  temperatura_c  umidade_relativa_pct
TMP-BG-20261020-013 2026-10-20 06:00  TMP-BG-01  Bangu           32.5                   NaN

Arquivo: pm25_tijuca_2026-10-11.csv

Colunas com valores ausentes:
pm25_ug_m3    1
dtype: int64

Registros afetados:
       id_registro        data_hora estacao_id bairro  pm25_ug_m3  pm10_ug_m3
AR-TJ-20261011-007 2026-10-11 03:00   AR-TJ-01 Tijuca         NaN       

## 9. Validação das informações ambientais

Além de identificar valores ausentes, precisamos verificar se os dados respeitam as regras de validade definidas para cada variável.

A função `validar_valores()` utiliza os seguintes critérios:

| Variável | Regra |
|---|---|
| Precipitação (mm) | Não pode ser negativa |
| Umidade relativa (%) | Entre 0 e 100 |
| PM2,5 (µg/m³) | Não pode ser negativo |
| PM10 (µg/m³) | Não pode ser negativo |

A função também identifica valores não numéricos nas colunas verificadas e informa quando uma coluna obrigatória para a regra está ausente.

**Interpretação do resultado**

Se somente o cabeçalho `VALIDAÇÃO DE VALORES` aparecer, nenhuma violação das regras implementadas foi encontrada.

Isso não significa que o conjunto de dados esteja completamente livre de inconsistências.

Os registros não são corrigidos ou excluídos automaticamente.


In [27]:
def validar_valores(pasta_destino):
    pasta_destino = Path(pasta_destino)

    regras = {
        "Chuvas": {
            "precipitacao_mm": (0, None)
        },
        "Temperatura": {
            "umidade_relativa_pct": (0, 100)
        },
        "Qualidade_do_Ar": {
            "pm25_ug_m3": (0, None),
            "pm10_ug_m3": (0, None)
        }
    }

    print("\nVALIDAÇÃO DE VALORES")
    print("=" * 50)
    lidos = 0
    falhas = []

    for categoria, colunas in regras.items():
        pasta = pasta_destino / categoria

        for arquivo in sorted(pasta.glob("*.csv")):
            try:
                dados = pd.read_csv(
                    arquivo, encoding="utf-8-sig",
                    dtype={"id_registro": "string", "estacao_id": "string"}
                )
            except UnicodeDecodeError as erro:
                falhas.append((categoria, arquivo.resolve()))
                print("\n[ERRO DE DECODIFICAÇÃO]")
                print(f"Arquivo: {arquivo.name}")
                print(f"Categoria: {categoria}")
                print(f"Caminho: {arquivo.resolve()}")
                print("Status: LEITURA NÃO CONCLUÍDA")
                print(f"Detalhes: {erro}")
                print("ORIENTAÇÃO: confira o formato do original e descubra sua codificação.")
                print("Crie uma cópia em UTF-8-SIG sem alterar o original; confira estrutura e registros.")
                print("Reprocesse a cópia pelo fluxo seguro do EcoData e confira a categoria.")
                print("Nenhuma alteração foi feita no arquivo. Continuando a análise...")
                continue
            except (pd.errors.ParserError, pd.errors.EmptyDataError, OSError) as erro:
                falhas.append((categoria, arquivo.resolve()))
                print("\n[ERRO DE LEITURA]")
                print(f"Arquivo: {arquivo.name}")
                print(f"Categoria: {categoria}")
                print(f"Caminho: {arquivo.resolve()}")
                print("Status: LEITURA NÃO CONCLUÍDA")
                print(f"Tipo: {type(erro).__name__}")
                print(f"Detalhes: {erro}")
                print("ORIENTAÇÃO: verifique a estrutura, o conteúdo e o acesso ao CSV.")
                print("Corrija uma cópia, valide sua leitura e reprocesse pelo fluxo seguro do EcoData.")
                print("Nenhuma alteração foi feita no arquivo. Continuando a análise...")
                continue

            lidos += 1

            for coluna, limites in colunas.items():
                if coluna not in dados.columns:
                    print(f"{arquivo.name}: coluna {coluna} ausente")
                    continue

                minimo, maximo = limites

                valores = pd.to_numeric(
                    dados[coluna],
                    errors="coerce"
                )

                nao_numericos = (
                    dados[coluna].notna() &
                    valores.isna()
                )

                invalidos = valores < minimo

                if maximo is not None:
                    invalidos = invalidos | (valores > maximo)

                problemas = invalidos | nao_numericos

                if problemas.any():
                    print(f"\nArquivo: {arquivo.name}")
                    print(f"Categoria: {categoria}")
                    print(f"Coluna: {coluna}")

                    print(f"Valores problemáticos: {problemas.sum()}")

                    if "id_registro" not in dados.columns:
                        print("\n[AVISO] COLUNA IDENTIFICADORA AUSENTE")
                        print("A coluna 'id_registro' não foi encontrada.")
                        print("Os registros problemáticos serão localizados pelo índice do DataFrame.")
                        print("Nenhum identificador foi criado e nenhuma medição foi alterada.")
                        registros = dados.loc[problemas, [coluna]].copy()
                        registros.index.name = "Índice do DataFrame"
                        print(registros.to_string(index=True))
                    else:
                        registros = dados.loc[
                            problemas,
                            ["id_registro", coluna]
                        ].copy()
                        id_sem_valor = (
                            registros["id_registro"].isna()
                            | registros["id_registro"]
                            .astype("string")
                            .str.strip()
                            .eq("")
                            .fillna(False)
                        )

                        if id_sem_valor.any():
                            print("\n[AVISO] IDENTIFICADOR INDISPONÍVEL")
                            print("Um ou mais valores de 'id_registro' estão nulos ou vazios.")
                            print(f"Identificadores indisponíveis: {id_sem_valor.sum()}")
                            print("O índice do DataFrame identifica essas ocorrências; nenhum ID foi alterado.")
                            registros.index.name = "Índice do DataFrame"
                            print(registros.to_string(index=True))
                        else:
                            print(registros.to_string(index=False))

    print("\nRESUMO DA LEITURA")
    print(f"Arquivos lidos com sucesso: {lidos}")
    print(f"Arquivos com erro de leitura: {len(falhas)}")
    if falhas:
        print("Arquivos não analisados:")
        for categoria_falha, caminho_falha in falhas:
            print(f"- {categoria_falha}: {caminho_falha}")

validar_valores(pasta_destino)



VALIDAÇÃO DE VALORES


# EcoData | Organização e qualidade de dados ambientais

## Objetivo e versão de referência

Organizar CSV/XLSX por categoria, preservar os arquivos recebidos e inspecionar a qualidade das informações ambientais. R2 e R3 foram consolidados a partir de `Automação_Ecodata_candidata_R1_R2_R3 copy.ipynb`, cujo código foi comparado com o commit `bc3623aec518199e77cd0b084bff2af561aeffad`. R4–R6 foram aplicados sobre essa base, com testes em diretórios temporários.

## Funcionalidades implementadas

- **R1 — Ingestão:** converte para um temporário na categoria, compara o CSV esperado com o existente e publica por hard link sem sobrescrever. Conteúdos divergentes são preservados em nomes `_origem_csv`, `_origem_xlsx` ou `_recuperado`, com sufixo numérico quando necessário; reexecuções reconhecem as saídas alternativas. A disponibilidade de hard links depende do sistema de arquivos; falhas são informadas e não autorizam sobrescrita.
- **EC-02 — Conteúdos arquivados:** a organização evita reimportar um conteúdo arquivado quando há uma cópia ativa correspondente. Se não houver cópia ativa, permite a recuperação. A presença de um hash no arquivo de backups, isoladamente, não bloqueia a recuperação.
- **R2 — Leitura:** `detalhar_ausentes()` e `validar_valores()` informam falhas de decodificação, estrutura CSV, arquivo vazio ou acesso, listam os não analisados e prosseguem para os demais. A contagem de leituras bem-sucedidas não representa aprovação integral da base.
- **R3 — Identificação:** medições problemáticas são apresentadas com `id_registro` quando disponível. Na ausência da coluna ou de um ID preenchido, o relatório usa o índice do DataFrame. Esse índice não é o número físico da linha do CSV. Nenhum ID é criado automaticamente.
- **R5 — Identificadores:** `id_registro` e `estacao_id` são textuais na conversão e nos relatórios. As medições mantêm a inferência e a validação numérica existente. IDs textuais Excel como `001` são preservados; números com formatação especial, datas ou fórmulas em colunas identificadoras exigem conferência antes da publicação. Não se reconstrói `001` a partir de `1` sem decisão sobre o esquema. A saída UTF-8-SIG usa CRLF explícito; a comparação da organização normaliza terminações LF/CRLF.
- **R6 — Classificação:** tokens completos do nome identificam Chuvas (`chuva`, `precipitacao`, `pluviometria`), Temperatura (`temperatura`, `temp`, `termometria`) e Qualidade_do_Ar (`ar`, `poluicao`, `pm25`). `qualidade` isolada não basta. Nomes desconhecidos, com pistas de várias categorias ou de ar junto de água/solo seguem para `Pendencias`; ambiguidades recebem aviso. Não se consulta o conteúdo para classificar nem se movem os arquivos existentes entre categorias.

## Significado dos relatórios

A organização conta arquivos publicados por categoria, ignorados, conflitos, migrações R5 pendentes e erros. Conflito não significa substituição: o arquivo anterior é preservado. Uma migração pendente exige revisão dos originais; o conteúdo não é declarado processado nem publicado automaticamente como nova informação independente.

A inspeção inicial informa linhas, colunas, valores ausentes e repetições internas por CSV. A investigação de ausentes mostra colunas e registros afetados. A validação separa valores não numéricos, limites ambientais e ausência de colunas verificadas. As regras são: precipitação não negativa, umidade entre 0 e 100, PM2,5 e PM10 não negativos. `NaN` não passa a ser violação de limite; continua sendo investigado na etapa de ausentes.

A verificação de duplicatas usa SHA-256 dos **bytes integrais**. A ingestão usa SHA-256 dos **campos CSV normalizados quanto às terminações de linha**, preservando diferenças textuais. Esses critérios não são equivalentes: arquivos com bytes diferentes podem representar os mesmos campos. Nenhum deles detecta sobreposição parcial de observações entre bases.

## Inventário histórico

O conjunto originalmente recebido continha **24 documentos CSV/XLSX**, com **754 registros incluindo as cópias redundantes**. Após a conversão inicial, eram 24 CSVs, dois arquivos redundantes e 22 conteúdos únicos:

| Categoria | Arquivos históricos | Registros incluindo redundâncias |
|:--|--:|--:|
| Chuvas | 7 | 228 |
| Temperatura | 7 | 234 |
| Qualidade_do_Ar | 6 | 184 |
| Pendencias | 4 | 108 |
| **Total** | **24** | **754** |

Os pares conhecidos eram `chuva_tijuca_2026-10-03.csv` / `precipitacao_tijuca_2026-10-03_repasse.csv` e `poluicao_centro_2026-10-05_repasse.csv` / `qualidade_ar_centro_2026-10-05.csv`. A escolha do arquivo ativo pode diferir da ordem em que esses pares foram identificados.

## Inventário verificado em 2026-10-24T01:20:30-03:00

Conferência direta, somente de leitura, dos arquivos presentes naquele momento; as contagens não foram extraídas das saídas históricas das células:

| Local/categoria | Arquivos | Registros |
|:--|--:|--:|
| `dados_recebidos` — originais CSV/XLSX | 24 | 754 |
| Ativos — Chuvas | 6 | 196 |
| Ativos — Temperatura | 7 | 234 |
| Ativos — Qualidade_do_Ar | 5 | 152 |
| Ativos — Pendencias | 4 | 108 |
| **Total ativo em `EcoData_Organizado`** | **22** | **690** |
| `EcoData_Duplicados_Arquivados` | 2 | 64 |

Os 22 CSVs ativos foram lidos nessa conferência, sem falhas; seus hashes binários são distintos. Somar 690 ativos e 64 arquivados reproduz os 754 registros do inventário com redundâncias, **não 754 observações independentes**.

Foram encontrados quatro valores ausentes: dois em `precipitacao_mm` de `chuva_meier_2026-10-07.csv`, um em `umidade_relativa_pct` de `temp_bangu_2026-10-20.csv` e um em `pm25_ug_m3` de `pm25_tijuca_2026-10-11.csv`. Não houve repetição interna exata de linhas nessa conferência. Nos arquivos das três categorias cobertas pelas regras ambientais, não foram encontrados valores fora dos limites, valores não numéricos nas medições verificadas nem ausência das colunas previstas. Os quatro arquivos de `Pendencias` não recebem essas regras de categoria. Isso não comprova que todos os aspectos dos dados estejam corretos.

Esses números descrevem uma observação datada. Novas entradas ou ações manuais podem mudar o inventário; confira os relatórios de uma nova execução antes de reutilizá-los como estado atual.

## Duplicatas, migração e segurança

O tratamento de duplicatas permanece em **`executar=False`**, sem movimentação ou remoção automática. Em uma ação manual com `executar=True`, a rotina move o redundante quando ainda não existe backup idêntico; se já existe, confirma o hash do backup e exclui somente a cópia redundante ativa. Os originais recebidos e os backups existentes são preservados. Nenhuma execução efetiva dessa rotina foi realizada para R4–R6.

R5 não migra o inventário antigo. Se a representação preservada do ID divergir de uma cópia histórica apenas em identificadores numericamente equivalentes, a organização informa **MIGRAÇÃO R5 NECESSÁRIA**, preserva o histórico e solicita conferência. Equivalência numérica não comprova equivalência de identificadores textuais; a regra é conservadora e pode exigir revisão de uma entrada legítima. Uma migração ou realocação de arquivos exige autorização separada.

## Verificações realizadas e limites

Os testes foram executados em cópias e dados fictícios temporários, sem recriar pastas de testes no projeto. Cobriram conflitos e publicações de R1, falhas de leitura de R2, identificação por índice de R3, zeros iniciais e ambiguidades Excel de R5, classificação R6 e reexecuções. As 24 conversões reais feitas para temporários mantiveram os bytes da conversão anterior para o inventário existente; uma cópia com 22 ativos e dois arquivados permaneceu estável em três reexecuções. Recuperação sem cópia ativa também foi verificada em cópia isolada.

Os resultados armazenados nas células foram preservados e podem pertencer a versões anteriores. Não foram regenerados sobre a base real. A presença de código ou de uma saída antiga não substitui a verificação da versão salva e dos resultados de teste correspondentes.

Limitações: esquema textual explícito restrito a `id_registro` e `estacao_id`; marcadores de ausência continuam seguindo o comportamento atual do Pandas; conversão Excel usa a primeira aba; nenhuma validação geral de esquema, qualidade científica, registros parcialmente repetidos ou recuperação automática de IDs formatados foi adicionada. Os testes cobrem os cenários descritos, sem garantir ausência de outros defeitos.

Próximas etapas possíveis, mediante autorização: revisar esquemas e casos de migração, reintroduzir testes permanentes se desejado e ampliar regras de qualidade justificadas. A versão ativa continua sendo `Automação_Ecodata.ipynb`; as cópias em `Versões Git` são referências históricas.
